<a href="https://colab.research.google.com/github/Ayushpal11/SAC/blob/main/Soft_Actor_Critic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Soft Actor Critic(SAC)**

Helps agent ny using past data and makes the model efficent(sample efficient) also it's an off Policy Model that's helpful in terms of helping Agent by maximazing rewards in terms of randomness plus entropy (randomness/diversity).

alpha -> Temperature

H -> Entropy

In [1]:
# Install dependencies for Box2D environments
!pip install swig
!pip install "gymnasium[box2d]"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 98.7 MB/s eta 0:00:00


In [2]:
# Importing the libraries
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

# Setting up the environment
env = gym.make("CarRacing-v3", render_mode='rgb_array')

# Setting the hyperparameters
# state_dim should be the flattened size of the observation space for image-based environments
state_dim = np.prod(env.observation_space.shape)
action_dim = env.action_space.shape[0]
hidden_dim = 256
actor_lr = 3e-4
critic_lr = 3e-4
alpha_lr = 3e-4
gamma = 0.99
tau = 0.005
buffer_size = 1e6
batch_size = 128
alpha = 0.2  # For Entropy coefficient

In [3]:
# Implementing the Replay Buffer
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=int(capacity))
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    def sample(self, batch_size):
        state, action, reward, next_state, done = zip(*random.sample(self.buffer, batch_size))
        return np.array(state), np.array(action), np.array(reward, dtype=np.float32), np.array(next_state), np.array(done, dtype=np.float32)
    def __len__(self):
        return len(self.buffer)


# Building the Actor Network
class Actor(nn.Module):
    def __init__(self):
        super(Actor, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.mean = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Linear(hidden_dim, action_dim)
    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = torch.relu(self.fc2(x))
        mean = self.mean(x)
        log_std = self.log_std(x)
        log_std = torch.clamp(log_std, min=-20, max=2)
        return mean, log_std
    def sample(self, state):
        mean, log_std = self.forward(state)
        std = log_std.exp()
        normal = torch.distributions.Normal(mean, std)
        x_t = normal.rsample()  # for reparameterization trick (mean + std * N(0,1))
        action = torch.tanh(x_t)

        # Computing log probability with tanh squashing correction
        log_prob = normal.log_prob(x_t)
        log_prob -= torch.log(1 - action.pow(2) + 1e-6)
        # Add small epsilon for numerical stability
        log_prob = log_prob.sum(dim=1, keepdim=True)
        # Sum log_probs across action dimensions

        return action, log_prob

# Building the Critic Network
class Critic(nn.Module):
    def __init__(self):
        super(Critic, self).__init__()
        self.fc1 = nn.Linear(state_dim + action_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)
    def forward(self, state, action):
        x = torch.cat([state, action], dim=1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Building the SAC Agent
class SACAgent:
    def __init__(self):
        super(SACAgent, self).__init__() # Call super constructor
        self.actor = Actor()
        self.critic_1 = Critic()
        self.critic_2 = Critic()
        self.target_critic_1 = Critic()
        self.target_critic_2 = Critic()
        self.target_critic_1.load_state_dict(self.critic_1.state_dict())
        self.target_critic_2.load_state_dict(self.critic_2.state_dict())
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=actor_lr)
        self.critic_1_optimizer = optim.Adam(self.critic_1.parameters(), lr=critic_lr)
        self.critic_2_optimizer = optim.Adam(self.critic_2.parameters(), lr=critic_lr)
        self.replay_buffer = ReplayBuffer(buffer_size)

    def select_action(self, state):
        state = torch.FloatTensor(state).reshape(1, -1) # Flatten the state
        action, _ = self.actor.sample(state) # Only need action for selection
        return action.detach().numpy()[0]

    def update(self, batch_size, gamma=gamma, tau=tau, alpha=alpha):
        if len(self.replay_buffer) < batch_size:
            return
        state, action, reward, next_state, done = self.replay_buffer.sample(batch_size)

        # Flatten states and next_states
        state = torch.FloatTensor(state).reshape(batch_size, -1)
        next_state = torch.FloatTensor(next_state).reshape(batch_size, -1)

        action = torch.FloatTensor(action)
        reward = torch.FloatTensor(reward).unsqueeze(1)
        done = torch.FloatTensor(np.float32(done)).unsqueeze(1)

        with torch.no_grad():
            next_state_action, next_state_log_pi = self.actor.sample(next_state)
            target_q1_next = self.target_critic_1(next_state, next_state_action)
            target_q2_next = self.target_critic_2(next_state, next_state_action)
            target_q_min = torch.min(target_q1_next, target_q2_next) - alpha * next_state_log_pi
            target_q = reward + (1 - done) * gamma * target_q_min

        # Update of the Critic 1 network
        current_q1 = self.critic_1(state, action)
        critic_1_loss = F.mse_loss(current_q1, target_q)
        self.critic_1_optimizer.zero_grad()
        critic_1_loss.backward()
        self.critic_1_optimizer.step()

        # Update of the Critic 2 network
        current_q2 = self.critic_2(state, action)
        critic_2_loss = F.mse_loss(current_q2, target_q)
        self.critic_2_optimizer.zero_grad()
        critic_2_loss.backward()
        self.critic_2_optimizer.step()

        # Update of the Actor network
        current_action, current_log_pi = self.actor.sample(state) # Get both action and log_pi for current state
        actor_loss = (alpha * current_log_pi - self.critic_1(state, current_action)).mean() # Correct SAC actor loss formulation
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        # Soft update of the Critic Target networks
        for target_param, param in zip(self.target_critic_1.parameters(), self.critic_1.parameters()):
            target_param.data.copy_(tau * param.data + (1 - tau) * target_param.data)
        for target_param, param in zip(self.target_critic_2.parameters(), self.critic_2.parameters()):
            target_param.data.copy_(tau * param.data + (1 - tau) * target_param.data)


# Creating the agent
agent = SACAgent()

In [ ]:
import torch.nn.functional as F

# Implementing the Training Loop
num_episodes = 100
for episode in range(num_episodes):
    # env.reset() now returns observation and info. We only need the observation.
    state, _ = env.reset()
    episode_reward = 0
    terminated = False
    truncated = False
    # The loop should continue as long as the episode is not terminated nor truncated
    while not terminated and not truncated:
        action = agent.select_action(state)
        # env.step() now returns observation, reward, terminated, truncated, and info
        next_state, reward, terminated, truncated, _ = env.step(action)

        # The 'done' signal for the replay buffer and update method usually combines terminated and truncated.
        done_signal = terminated or truncated
        agent.replay_buffer.push(state, action, reward, next_state, done_signal);
        agent.update(batch_size)
        state = next_state
        episode_reward += reward
    print(f"Episode {episode}: Total Reward: {episode_reward}")

In [ ]:
import glob
import io
import base64
import imageio
from IPython.display import HTML, display

def show_video_of_model(agent, env):
  state, _ = env.reset()
  done = False
  frames = []
  while not done:
    frame = env.render()
    frames.append(frame)

    # Flatten the state before passing it to the agent's select_action method
    flat_state = state.flatten()
    action = agent.select_action(flat_state)

    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
  env.close()
  imageio.mimsave('video.mp4', frames, fps=30)

def show_video():
    mp4list = glob.glob('*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

# Call the functions to generate and display the video
show_video_of_model(agent, env)
show_video()